## **Project Title:**
### **"Supervised Deep Neural Networks for Multi-Disease Medical Image Classification in Federated Learning Settings"**

#### **Group Members:**
` Misbah Khan`
`Wajeeha Mahmood`

In [ ]:

import os
import copy
import csv
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm

warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────
# 1. CONFIGURATION
# ──────────────────────────────────────────────

@dataclass
class Config:
    # ── Dataset ──────────────────────────────
    # Choose: "tb_xray" | "brain_tumor" | "diabetic_retinopathy"
    dataset_name: str = "brain_tumor"
    data_root: str = "./data"                    # root directory for all datasets

    # ── Model ─────────────────────────────────
    # Choose: "resnet50" | "vgg16" | "efficientnet_b0"
    backbone: str = "resnet50"
    img_size: int = 224
    pretrained: bool = True

    # ── Federated Learning ────────────────────
    num_clients: int = 10
    num_rounds: int = 30                         # global communication rounds
    local_epochs: int = 3                        # local epochs per round
    local_batch_size: int = 32
    fraction_fit: float = 1.0                    # fraction of clients per round

    # ── Adaptive Aggregation ──────────────────
    divergence_threshold_init: float = 0.10      # τ — paper's value
    threshold_ema_alpha: float = 0.2             # EMA smoothing for adaptive τ
    fedprox_mu: float = 0.01                     # proximal term weight (improvement)

    # ── Optimizer ─────────────────────────────
    lr: float = 1e-3
    weight_decay: float = 1e-4
    warmup_rounds: int = 3                       # linear LR warmup

    # ── Non-IID Partitioning ──────────────────
    dirichlet_alpha: float = 0.5                 # lower = more non-IID
    # Clients 2, 5, 8 get extra class imbalance (mirrors paper)
    imbalanced_clients: List[int] = field(default_factory=lambda: [2, 5, 8])
    imbalance_ratio: float = 0.1                 # minority class fraction

    # ── Label Smoothing ───────────────────────
    label_smoothing: float = 0.1

    # ── Mixup ─────────────────────────────────
    use_mixup: bool = True
    mixup_alpha: float = 0.4

    # ── Train / Val / Test split ──────────────
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    # test = remaining 0.15

    # ── Misc ──────────────────────────────────
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./checkpoints"
    log_every: int = 5                           # print metrics every N rounds


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ──────────────────────────────────────────────
# 2. DATASET PATHS & TRANSFORMS
# ──────────────────────────────────────────────

DATASET_PATHS = {
    "tb_xray": {
        "train": "TB_Chest_Radiography_Database",   # ← was TB_Chest_Xray_Dataset
        "test":  None,
    },
    "brain_tumor": {
        "train": "Training",                         # ✓ correct
        "test":  "Testing",                          # ✓ correct
    },
    "diabetic_retinopathy": {
        "train": "colored_images",                   # ✓ correct
        "test":  None,
    },
}


def get_transforms(img_size: int, split: str):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    if split == "train":
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2),     # after ToTensor
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

from torchvision.datasets import ImageFolder
from PIL import Image, UnidentifiedImageError

class SafeImageFolder(ImageFolder):
    def __getitem__(self, index):
        try:
            return super().__getitem__(index)
        except (UnidentifiedImageError, OSError):
            # Return a black image with label 0 if file is corrupted
            img = Image.new('RGB', (self.transform.transforms[0].size
                  if hasattr(self.transform.transforms[0], 'size')
                  else (224, 224)), (0, 0, 0))
            if self.transform:
                img = self.transform(img)
            return img, 0

def load_full_dataset(cfg: Config) -> Tuple[datasets.ImageFolder, datasets.ImageFolder]:
    """Returns (train_dataset, test_dataset). If no explicit test split,
    test_dataset is a val-transformed view of the held-out portion."""
    info = DATASET_PATHS[cfg.dataset_name]
    train_path = os.path.join(cfg.data_root, info["train"])

    # Training dataset with augmentation
    full_train = SafeImageFolder(train_path, transform=get_transforms(cfg.img_size, "train"))

    if info["test"] is not None:
        test_path  = os.path.join(cfg.data_root, info["test"])
        test_ds    = SafeImageFolder(test_path, transform=get_transforms(cfg.img_size, "val"))
        return full_train, test_ds
    else:
        # Split off test from train
        n = len(full_train)
        indices = list(range(n))
        labels  = [full_train.targets[i] for i in indices]
        train_idx, test_idx = train_test_split(
            indices, test_size=(1 - cfg.train_ratio - cfg.val_ratio) * 2,
            stratify=labels, random_state=cfg.seed
        )
        # Re-create test with eval transforms
        test_base = SafeImageFolder(train_path, transform=get_transforms(cfg.img_size, "val"))
        train_subset = Subset(full_train, train_idx)
        test_subset  = Subset(test_base,  test_idx)
        return train_subset, test_subset


# ──────────────────────────────────────────────
# 3. NON-IID DIRICHLET PARTITION
# ──────────────────────────────────────────────

def dirichlet_partition(
    targets: List[int],
    num_clients: int,
    alpha: float,
    imbalanced_clients: List[int],
    imbalance_ratio: float,
    seed: int
) -> Dict[int, List[int]]:
    """
    Partition indices among clients using Dirichlet distribution.
    Clients in imbalanced_clients get one class severely undersampled.
    Returns {client_id: [indices]}.
    """
    np.random.seed(seed)
    targets = np.array(targets)
    num_classes = len(np.unique(targets))
    class_indices = {c: np.where(targets == c)[0].tolist() for c in range(num_classes)}

    # Shuffle within each class
    for c in class_indices:
        random.shuffle(class_indices[c])

    client_indices = defaultdict(list)

    for c, idx_list in class_indices.items():
        proportions = np.random.dirichlet([alpha] * num_clients)
        proportions = proportions / proportions.sum()
        splits = (proportions * len(idx_list)).astype(int)
        splits[-1] = len(idx_list) - splits[:-1].sum()   # fix rounding

        ptr = 0
        for k in range(num_clients):
            client_indices[k].extend(idx_list[ptr: ptr + splits[k]])
            ptr += splits[k]

    # Apply extra imbalance to specified clients (mirror paper's setup)
    for k in imbalanced_clients:
        idx = client_indices[k]
        idx_targets = targets[idx]
        # Pick the majority class and undersample it
        classes, counts = np.unique(idx_targets, return_counts=True)
        if len(classes) < 2:
            continue
        majority = classes[np.argmax(counts)]
        maj_idx = [i for i in idx if targets[i] == majority]
        rest    = [i for i in idx if targets[i] != majority]
        keep_n  = max(1, int(len(maj_idx) * imbalance_ratio))
        client_indices[k] = random.sample(maj_idx, keep_n) + rest

    return dict(client_indices)


def get_targets(dataset) -> List[int]:
    """Extract targets/labels from a dataset or Subset."""
    if isinstance(dataset, Subset):
        full_targets = dataset.dataset.targets
        return [full_targets[i] for i in dataset.indices]
    return dataset.targets


# ──────────────────────────────────────────────
# 4. MODEL BUILDER
# ──────────────────────────────────────────────

def build_model(backbone: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    if backbone == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    elif backbone == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1" if pretrained else None)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    elif backbone == "efficientnet_b0":
        try:
            import timm
            model = timm.create_model("efficientnet_b0", pretrained=pretrained,
                                      num_classes=num_classes)
        except ImportError:
            raise ImportError("pip install timm  to use EfficientNet-B0")

    else:
        raise ValueError(f"Unknown backbone: {backbone}")

    # Freeze only early layers, unfreeze last 2 blocks + head
    for name, param in model.named_parameters():
        if any(x in name for x in ["layer3", "layer4", "fc"]):
            param.requires_grad = True
        else:
            param.requires_grad = False
    return model


def get_model_params(model: nn.Module) -> List[torch.Tensor]:
    return [p.data.clone() for p in model.parameters()]


def set_model_params(model: nn.Module, params: List[torch.Tensor]):
    for p, new_p in zip(model.parameters(), params):
        p.data.copy_(new_p)


# ──────────────────────────────────────────────
# 5. LOSS FUNCTION
# ──────────────────────────────────────────────

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing: float = 0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            smooth = torch.full_like(log_probs, self.smoothing / (n_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        return -(smooth * log_probs).sum(dim=-1).mean()


# ──────────────────────────────────────────────
# 6. MIXUP
# ──────────────────────────────────────────────

def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ──────────────────────────────────────────────
# 7. LOCAL TRAINING (FedProx-aware)
# ──────────────────────────────────────────────

def local_train(
    model: nn.Module,
    global_params: List[torch.Tensor],
    loader: DataLoader,
    cfg: Config,
    use_proximal: bool = False,
) -> Tuple[List[torch.Tensor], float]:
    """
    Train model locally for cfg.local_epochs.
    If use_proximal=True, add FedProx term: (μ/2)||w - w_global||²
    Returns (updated_params, avg_train_loss).
    """
    model.train()
    criterion = LabelSmoothingCrossEntropy(cfg.label_smoothing).to(cfg.device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.local_epochs, eta_min=cfg.lr * 0.1)

    total_loss = 0.0
    steps = 0

    for _ in range(cfg.local_epochs):
        for batch in loader:
            imgs, labels = batch
            imgs, labels = imgs.to(cfg.device), labels.to(cfg.device)

            if cfg.use_mixup:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels, cfg.mixup_alpha)
                outputs = model(imgs)
                loss = mixup_loss(criterion, outputs, y_a, y_b, lam)
            else:
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            # FedProx proximal term
            if use_proximal:
                prox = sum(
                    torch.norm(p - gp.to(cfg.device)) ** 2
                    for p, gp in zip(model.parameters(), global_params)
                )
                loss = loss + (cfg.fedprox_mu / 2) * prox

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # gradient clip
            optimizer.step()

            total_loss += loss.item()
            steps += 1

        scheduler.step()

    updated_params = get_model_params(model)
    return updated_params, total_loss / max(steps, 1)


# ──────────────────────────────────────────────
# 8. DIVERGENCE & ADAPTIVE AGGREGATION
# ──────────────────────────────────────────────

def compute_divergence(client_params_list, global_params):
    total_params = sum(p.numel() for p in global_params)
    deltas = []
    for client_params in client_params_list:
        dist = sum(
            torch.norm(cp.float() - gp.float()).item() ** 2
            for cp, gp in zip(client_params, global_params)
        )
        deltas.append((dist ** 0.5) / (total_params ** 0.5))  # normalize
    return float(np.mean(deltas))


def fedavg_aggregate(
    client_params_list: List[List[torch.Tensor]],
    client_weights: List[float],
) -> List[torch.Tensor]:
    """Weighted FedAvg: w_global = Σ α_k * w_k."""
    total = sum(client_weights)
    new_params = []
    for layer_idx in range(len(client_params_list[0])):
        agg = sum(
            (w / total) * cp[layer_idx].float()
            for cp, w in zip(client_params_list, client_weights)
        )
        new_params.append(agg)
    return new_params


def fedsgd_aggregate(
    client_params_list: List[List[torch.Tensor]],
    global_params: List[torch.Tensor],
    client_weights: List[float],
    lr: float,
) -> List[torch.Tensor]:
    """
    FedSGD: w_global ← w_global - η * Σ (α_k * grad_k)
    grad_k ≈ w_global - w_k  (negative local update direction)
    """
    total = sum(client_weights)
    new_params = []
    for layer_idx in range(len(global_params)):
        weighted_grad = sum(
            (w / total) * (global_params[layer_idx].float() - cp[layer_idx].float())
            for cp, w in zip(client_params_list, client_weights)
        )
        new_params.append(global_params[layer_idx].float() - lr * weighted_grad)
    return new_params


def adaptive_aggregate(
    client_params_list: List[List[torch.Tensor]],
    global_params: List[torch.Tensor],
    client_weights: List[float],
    divergence: float,
    threshold: float,
    lr: float,
) -> Tuple[List[torch.Tensor], str]:
    """
    Selects FedAvg or FedSGD based on divergence vs threshold.
    Returns (aggregated_params, method_used).
    """
    if divergence > threshold:
        new_params = fedsgd_aggregate(client_params_list, global_params, client_weights, lr)
        return new_params, "FedSGD"
    else:
        new_params = fedavg_aggregate(client_params_list, client_weights)
        return new_params, "FedAvg"


# ──────────────────────────────────────────────
# 9. EVALUATION
# ──────────────────────────────────────────────

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str) -> Tuple[float, float]:
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds) * 100
    f1  = f1_score(all_labels, all_preds, average="weighted") * 100
    return acc, f1


@torch.no_grad()
def evaluate_with_report(model: nn.Module, loader: DataLoader,
                          device: str, class_names: List[str]) -> str:
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return classification_report(all_labels, all_preds, target_names=class_names)


# ──────────────────────────────────────────────
# 10. MAIN FEDERATED LEARNING LOOP
# ──────────────────────────────────────────────

def federated_learning(cfg: Config):
    set_seed(cfg.seed)
    os.makedirs(cfg.save_dir, exist_ok=True)
    device = cfg.device
    print(f"\n{'='*60}")
    print(f"  Federated Learning | {cfg.backbone.upper()} | {cfg.dataset_name}")
    print(f"  Device: {device} | Clients: {cfg.num_clients} | Rounds: {cfg.num_rounds}")
    print(f"{'='*60}\n")

    # ── Load data ──────────────────────────────
    print("Loading dataset...")
    train_dataset, test_dataset = load_full_dataset(cfg)
    num_classes = len(train_dataset.dataset.classes if isinstance(train_dataset, Subset)
                      else train_dataset.classes)
    class_names = (train_dataset.dataset.classes if isinstance(train_dataset, Subset)
                   else train_dataset.classes)
    print(f"Classes ({num_classes}): {class_names}")

    # ── Partition for FL ───────────────────────
    targets = get_targets(train_dataset)
    client_idx_map = dirichlet_partition(
        targets, cfg.num_clients, cfg.dirichlet_alpha,
        cfg.imbalanced_clients, cfg.imbalance_ratio, cfg.seed
    )

    # Build per-client train loaders
    # Val loader from each client's held-out 15%
    client_train_loaders = {}
    client_val_loaders   = {}
    client_sizes         = {}

    for k, idx_list in client_idx_map.items():
        if len(idx_list) < 4:
            idx_list = idx_list * 4          # pad tiny client
        val_n   = max(1, int(len(idx_list) * cfg.val_ratio))
        train_n = len(idx_list) - val_n
        train_idx_k = idx_list[:train_n]
        val_idx_k   = idx_list[train_n:]

        # Build index subset against the original dataset
        if isinstance(train_dataset, Subset):
            base_train_idx = [train_dataset.indices[i] for i in train_idx_k]
            base_val_idx   = [train_dataset.indices[i] for i in val_idx_k]
            train_sub = Subset(train_dataset.dataset, base_train_idx)
            val_sub   = Subset(train_dataset.dataset, base_val_idx)
            # Val needs eval transforms — rebuild subset with eval transforms
            val_base  = datasets.ImageFolder(
                os.path.join(cfg.data_root, DATASET_PATHS[cfg.dataset_name]["train"]),
                transform=get_transforms(cfg.img_size, "val")
            )
            val_sub = Subset(val_base, base_val_idx)
        else:
            train_sub = Subset(train_dataset, train_idx_k)
            val_sub   = Subset(train_dataset, val_idx_k)

        client_train_loaders[k] = DataLoader(
            train_sub, batch_size=cfg.local_batch_size, shuffle=True,
            num_workers=0, pin_memory=(device == "cuda"), drop_last=True
        )
        client_val_loaders[k] = DataLoader(
            val_sub, batch_size=64, shuffle=False,
            num_workers=0, pin_memory=(device == "cuda")
        )
        client_sizes[k] = len(train_idx_k)

    test_loader = DataLoader(
        test_dataset, batch_size=64, shuffle=False,
        num_workers=0, pin_memory=(device == "cuda")
    )

    # Print distribution summary
    print(f"\nClient data sizes: {[client_sizes[k] for k in range(cfg.num_clients)]}")
    total_train = sum(client_sizes.values())
    print(f"Total training samples: {total_train}")

    # ── Initialize global model ────────────────
    global_model = build_model(cfg.backbone, num_classes, cfg.pretrained).to(device)
    global_params = get_model_params(global_model)

    # ── Tracking ───────────────────────────────
    history = {
        "round": [], "test_acc": [], "test_f1": [],
        "divergence": [], "method": [], "threshold": []
    }
    best_acc = 0.0
    threshold = cfg.divergence_threshold_init   # adaptive τ

    # ── FL Rounds ──────────────────────────────
    for rnd in range(1, cfg.num_rounds + 1):

        # Linear warmup on effective lr
        warmup_factor = min(rnd / cfg.warmup_rounds, 1.0)
        eff_lr = cfg.lr * warmup_factor

        # Select fraction of clients
        n_fit = max(1, int(cfg.num_clients * cfg.fraction_fit))
        selected = random.sample(range(cfg.num_clients), n_fit)

        # ── Client local training ──────────────
        client_params_list  = []
        client_weight_list  = []
        client_val_acc_list = []
        client_loss_list    = []

        for k in selected:
            # Clone global model for this client
            local_model = build_model(cfg.backbone, num_classes, False).to(device)
            set_model_params(local_model, global_params)

            # Decide FedProx: use when high divergence anticipated
            use_prox = (rnd > cfg.warmup_rounds)

            updated_params, train_loss = local_train(
                local_model, global_params,
                client_train_loaders[k], cfg,
                use_proximal=use_prox
            )
            val_acc, _ = evaluate(local_model, client_val_loaders[k], device)

            client_params_list.append(updated_params)
            client_val_acc_list.append(val_acc)
            client_weight_list.append(client_sizes[k])
            client_loss_list.append(train_loss)

        # ── Compute divergence ─────────────────
        divergence = compute_divergence(client_params_list, global_params)

        # Improvement: weight by val accuracy (not just dataset size)
        # w_k = n_k * softmax(val_acc_k)
        acc_arr  = np.array(client_val_acc_list)
        acc_soft = np.exp(acc_arr / 10) / np.exp(acc_arr / 10).sum()   # softmax
        size_arr = np.array(client_weight_list, dtype=float)
        combined_weights = list(size_arr / size_arr.sum() * 0.5 + acc_soft * 0.5)

        # ── Adaptive aggregation ───────────────
        new_params, method = adaptive_aggregate(
            client_params_list, global_params,
            combined_weights, divergence, threshold, eff_lr
        )

        # Adaptive threshold EMA update
        threshold = (1 - cfg.threshold_ema_alpha) * threshold + \
                     cfg.threshold_ema_alpha * divergence

        # Update global model
        global_params = [p.clone() for p in new_params]
        set_model_params(global_model, global_params)

        # ── Evaluation ─────────────────────────
        test_acc, test_f1 = evaluate(global_model, test_loader, device)

        history["round"].append(rnd)
        history["test_acc"].append(test_acc)
        history["test_f1"].append(test_f1)
        history["divergence"].append(divergence)
        history["method"].append(method)
        history["threshold"].append(threshold)

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(global_model.state_dict(),
                       os.path.join(cfg.save_dir, "best_global_model.pt"))

        if rnd % cfg.log_every == 0 or rnd == 1:
            avg_loss = np.mean(client_loss_list)
            print(f"[Round {rnd:3d}/{cfg.num_rounds}]  "
                  f"Acc={test_acc:.2f}%  F1={test_f1:.2f}%  "
                  f"Div={divergence:.4f}  τ={threshold:.4f}  "
                  f"Method={method}  Loss={avg_loss:.4f}  "
                  f"Best={best_acc:.2f}%")

    # ── Final evaluation ───────────────────────
    print(f"\n{'='*60}")
    print(f"  Training complete. Best test accuracy: {best_acc:.2f}%")
    print(f"{'='*60}\n")

    # Load best model and print classification report
    global_model.load_state_dict(
        torch.load(os.path.join(cfg.save_dir, "best_global_model.pt"),
                   map_location=device)
    )
    report = evaluate_with_report(global_model, test_loader, device, class_names)
    print("Classification Report (best model):\n")
    print(report)

    return history, global_model


# ──────────────────────────────────────────────
# 11. CENTRALIZED BASELINE (Differential Privacy)
# ──────────────────────────────────────────────

def add_dp_noise(model: nn.Module, sigma: float = 0.5):
    """Add Gaussian noise to gradients: ∇L ← ∇L + N(0, σ²)."""
    with torch.no_grad():
        for p in model.parameters():
            if p.grad is not None:
                p.grad += torch.randn_like(p.grad) * sigma


def centralized_baseline(cfg: Config, dp_sigma: float = 0.5, epochs: int = 20):
    """Full dataset training with differential privacy (paper's centralized comparison)."""
    set_seed(cfg.seed)
    device = cfg.device
    print(f"\n{'='*60}")
    print(f"  Centralized Baseline (DP σ={dp_sigma}) | {cfg.backbone.upper()}")
    print(f"{'='*60}\n")

    train_dataset, test_dataset = load_full_dataset(cfg)
    num_classes = len(train_dataset.dataset.classes if isinstance(train_dataset, Subset)
                      else train_dataset.classes)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                              num_workers=0, pin_memory=(device == "cuda"))
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                              num_workers=0, pin_memory=(device == "cuda"))

    model     = build_model(cfg.backbone, num_classes, cfg.pretrained).to(device)
    criterion = LabelSmoothingCrossEntropy(cfg.label_smoothing).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=cfg.lr * 0.01)

    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            add_dp_noise(model, dp_sigma)          # differential privacy
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        scheduler.step()

        acc, f1 = evaluate(model, test_loader, device)
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(),
                       os.path.join(cfg.save_dir, "best_centralized_model.pt"))

        if epoch % 5 == 0:
            print(f"  [Epoch {epoch:3d}] Acc={acc:.2f}%  F1={f1:.2f}%  Best={best_acc:.2f}%")

    print(f"\n  Centralized Best Accuracy: {best_acc:.2f}%\n")
    return best_acc


# ──────────────────────────────────────────────
# 12. QUICK SUMMARY PRINTER
# ──────────────────────────────────────────────

def print_summary(history: dict):
    rounds   = history["round"]
    accs     = history["test_acc"]
    f1s      = history["test_f1"]
    methods  = history["method"]

    n_fedavg = methods.count("FedAvg")
    n_fedsgd = methods.count("FedSGD")

    print(f"\n{'─'*50}")
    print(f"  FL Summary")
    print(f"{'─'*50}")
    print(f"  Final  Acc : {accs[-1]:.2f}%")
    print(f"  Best   Acc : {max(accs):.2f}%  (round {rounds[accs.index(max(accs))]})")
    print(f"  Final  F1  : {f1s[-1]:.2f}%")
    print(f"  FedAvg used: {n_fedavg} rounds ({n_fedavg/len(rounds)*100:.0f}%)")
    print(f"  FedSGD used: {n_fedsgd} rounds ({n_fedsgd/len(rounds)*100:.0f}%)")
    print(f"{'─'*50}\n")


# ──────────────────────────────────────────────
# 13. ENTRY POINT
# ──────────────────────────────────────────────

def save_logs(history, path="./logs_brain_tumor.csv"):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=history.keys())
        writer.writeheader()
        rows = [dict(zip(history.keys(), vals))
                for vals in zip(*history.values())]
        writer.writerows(rows)
    print(f"Logs saved to {path}")

def main():
    cfg = Config(
    dataset_name = "brain_tumor", #For Brain Tumor Dataset
    # dataset_name = "tb_xray",    #For Tuberculosis Dataset
    # dataset_name = "diabetic_retinopathy", #For Diabetic Retinopathy Dataset
    data_root           = "/content/drive/MyDrive/fl_data",
    save_dir            = "/content/drive/MyDrive/fl_checkpoints",
    backbone            = "resnet50",
    num_rounds          = 40,          # more rounds
    local_epochs        = 2,           # was 5 — key fix for drift
    num_clients         = 5,
    lr                  = 2e-4,        # lower
    dirichlet_alpha     = 2.5,         # more balanced across clients
    imbalanced_clients  = [],          # disable extra imbalance
    divergence_threshold_init = 0.002,
    use_mixup           = False,
)
      # ── Federated training ─────────────────────
    history, fl_model = federated_learning(cfg)
    # Call after federated_learning() returns
    save_logs(history,"./logs_brain_tumor.csv")  #For Brain Tumor Log File
    # save_logs(history, "./logs_tb.csv")  #For Tuberculosis Log File
    # save_logs(history, "./logs_dr.csv")  #For Diabetic Retinopathy Log File
    print_summary(history)

    # ── Optional: centralized baseline ─────────
    # cent_acc = centralized_baseline(cfg, dp_sigma=0.5, epochs=20)

    return history, fl_model


if __name__ == "__main__":
    main()



  Federated Learning | RESNET50 | tb_xray
  Device: cuda | Clients: 5 | Rounds: 40

Loading dataset...
Classes (2): ['Normal', 'Tuberculosis']

Client data sizes: [573, 350, 387, 368, 822]
Total training samples: 2500
[Round   1/40]  Acc=83.35%  F1=75.78%  Div=0.0010  τ=0.0018  Method=FedAvg  Loss=0.4290  Best=83.35%
[Round   5/40]  Acc=83.35%  F1=75.78%  Div=0.0005  τ=0.0012  Method=FedAvg  Loss=0.3778  Best=83.35%
[Round  10/40]  Acc=83.35%  F1=75.78%  Div=0.0005  τ=0.0007  Method=FedAvg  Loss=0.3699  Best=83.35%
[Round  15/40]  Acc=83.35%  F1=75.78%  Div=0.0005  τ=0.0006  Method=FedAvg  Loss=0.3636  Best=83.35%
